# Comparable training: ODRA vs simulator (trimmed reverse entanglement, Z on q0)

Same pipeline as [`train_comparable_ansatze.ipynb`](train_comparable_ansatze.ipynb): shared banknote data, `StatevectorEstimator`, **`ParamShiftEstimatorGradient`**.

**Readout:** Pauli **Z** on **`q_0`** (`MEASURE_QUBIT = 0` → `"IIIIZ"` for 5 qubits).

**Ansatz change:** In the **reverse** entangling sub-layer only, keep **only the two ring edges incident on qubit 0** (`(0, n-1)` and `(1, 0)`). Drop reverse pairs that do not involve `q_0` (for 5 qubits: drop `(2,1)`, `(3,2)`, `(4,3)` and their associated Ry+CZ / CRY parameters). The **forward** ring is unchanged.

**Parameters:** The reverse sub-layer is trimmed to **q0-incident edges only** (`3n + 2` parameters **in that macro-layer**) **only on the last** macro-layer before readout. Earlier macro-layers keep the **full** reverse ring (`4n` per macro-layer), so entanglement can still reach `q_0` through the rest of the circuit. Example: `n=5`, depth `2` → **17** weights; depth `4` → **37** (= 20 + 17).

**Outputs** (filenames avoid clobbering the full ansatz notebook):

- `test_data_trimmed_reverse_q0.csv`
- `weights_odra_trimmed_reverse_q0.csv`, `weights_simulator_trimmed_reverse_q0.csv` — **best** checkpoint (`param_index`, `param_name`, `value`)
- `weights_odra_best_trimmed_reverse_q0.pth`, `weights_odra_final_trimmed_reverse_q0.pth`, `weights_simulator_best_trimmed_reverse_q0.pth`, `weights_simulator_final_trimmed_reverse_q0.pth`

**Reproducibility:** `RANDOM_SEED` drives Python `random`, NumPy, PyTorch (CPU/CUDA), `train_test_split`, `DataLoader` shuffle, and each `train_and_save` re-seeds before model construction. CUDA uses deterministic cuDNN where applicable.


In [21]:
import csv
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
from ucimlrepo import fetch_ucirepo

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient
from qiskit_machine_learning.neural_networks import EstimatorQNN

ROOT = Path.cwd().resolve()
for parent in [ROOT, *ROOT.parents]:
    if (parent / "pyproject.toml").is_file():
        ROOT = parent
        break

SETUP_DIR = ROOT / "tests" / "ansatz_odra_evaluation" / "setup"
SETUP_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_QUBITS = 5
ANSATZ_DEPTH = 4
MEASURE_QUBIT = 0  # Z on q_0
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.01

# Suffix for all artifacts from this notebook
ARTIFACT_SUFFIX = "_trimmed_reverse_q0"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"ROOT={ROOT}")
print(f"SETUP_DIR={SETUP_DIR}")
print(f"MEASURE_QUBIT={MEASURE_QUBIT}, depth={ANSATZ_DEPTH}, artifact suffix={ARTIFACT_SUFFIX!r}")


ROOT=/Users/jkw/Documents/uni/axion/QC1
SETUP_DIR=/Users/jkw/Documents/uni/axion/QC1/tests/ansatz_odra_evaluation/setup
MEASURE_QUBIT=0, depth=4, artifact suffix='_trimmed_reverse_q0'


In [22]:
def prepare_data(test_size: float = 0.2, random_state: int = 42):
    banknote_authentication = fetch_ucirepo(id=267)
    X = banknote_authentication.data.features.to_numpy()
    y = banknote_authentication.data.targets.to_numpy().ravel()
    indices = np.arange(len(X))

    assert X.shape[1] == 4, f"Expected 4 features, got {X.shape[1]}"
    assert set(np.unique(y)) == {0, 1}, f"Expected binary labels {{0, 1}}, got {set(np.unique(y))}"

    variance = X[:, 0].reshape(-1, 1)
    skewness = X[:, 1].reshape(-1, 1)
    interaction = variance * skewness
    X_expanded = np.hstack((X, interaction))

    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X_expanded, y, indices, test_size=test_size, random_state=random_state
    )

    scaler = MinMaxScaler(feature_range=(-np.pi / 4, np.pi / 4))
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Dataset loaded: {len(X_train)} train, {len(X_test)} test samples")
    print(f"Feature dimension: {X_train_scaled.shape[1]}")

    return X_train_scaled, X_test_scaled, y_train, y_test, idx_train, idx_test


def pauli_z_string(num_qubits: int, z_qubit_index: int) -> str:
    if not 0 <= z_qubit_index < num_qubits:
        raise ValueError(f"z_qubit_index must be in [0, {num_qubits}), got {z_qubit_index}")
    chars = ["I"] * num_qubits
    chars[num_qubits - 1 - z_qubit_index] = "Z"
    return "".join(chars)


X_train, X_test, y_train_raw, y_test_raw, train_idx, test_idx = prepare_data(
    test_size=0.2, random_state=RANDOM_SEED
)

pauli_label = pauli_z_string(N_QUBITS, MEASURE_QUBIT)
print(f"Observable Pauli string: {pauli_label!r}")


Dataset loaded: 1097 train, 275 test samples
Feature dimension: 5
Observable Pauli string: 'IIIIZ'


In [ ]:
def ansatz_trimmed_reverse_q0_param_count(n_qubits: int, depth: int) -> int:
    """Weights when only the last macro-layer uses the q0-incident reverse trim."""
    m = depth // 2
    if m == 0:
        return 0
    full = 4 * n_qubits
    last = 3 * n_qubits + 2
    return (m - 1) * full + last


def odra_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    # Full reverse (4n) on all macro-layers except the last; there trim reverse to q0 edges only (3n+2).
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    p = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[p + i], i)
        p += n_qubits

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.rz(theta[p + i], target)
            qc.cz(control, target)
        p += n_qubits

        for i in range(n_qubits):
            qc.rx(theta[p + i], i)
        p += n_qubits

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + k], target)
                qc.cz(control, target)
            p += 2
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + i], target)
                qc.cz(control, target)
            p += n_qubits

    assert p == len(theta)
    return qc


def simulator_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    param_idx = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[param_idx], i)
            param_idx += 1

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.crx(theta[param_idx], control, target)
            param_idx += 1

        for i in range(n_qubits):
            qc.rx(theta[param_idx], i)
            param_idx += 1

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1

    assert param_idx == len(theta)
    return qc


In [24]:
class HybridModel(nn.Module):
    def __init__(self, ansatz_circuit: QuantumCircuit, num_qubits: int, z_qubit_index: int):
        super().__init__()
        self.feature_map = self._create_angle_encoding(num_qubits)
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)

        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)

        obs_str = pauli_z_string(num_qubits, z_qubit_index)
        observable = SparsePauliOp.from_list([(obs_str, 1.0)])

        estimator = StatevectorEstimator()
        gradient = ParamShiftEstimatorGradient(estimator=estimator)
        self.qnn = EstimatorQNN(
            circuit=self.qc,
            observables=observable,
            input_params=input_params,
            weight_params=weight_params,
            estimator=estimator,
            gradient=gradient,
        )
        self.quantum_layer = TorchConnector(self.qnn)

    def _create_angle_encoding(self, num_qubits: int) -> QuantumCircuit:
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector("x", num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.quantum_layer(x)


In [25]:
y_train = 2 * y_train_raw - 1
y_test_pm = 2 * y_test_raw - 1

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_pm, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)


def make_train_loader():
    g = torch.Generator()
    g.manual_seed(RANDOM_SEED)
    return DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=g)


train_loader = make_train_loader()
loss_function = nn.MSELoss()


def train_epoch(model, train_loader, optimizer, loss_fn):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        num_batches += 1
    return epoch_loss / num_batches


def evaluate(model, X_tensor, y_tensor, loss_fn):
    model.eval()
    with torch.no_grad():
        outputs = model(X_tensor)
        loss = loss_fn(outputs, y_tensor).item()
        predicted = (outputs > 0).float() * 2 - 1
        correct = (predicted == y_tensor).sum().item()
        accuracy = correct / len(y_tensor)
    return {"loss": loss, "accuracy": accuracy}


def save_test_csv():
    path = SETUP_DIR / f"test_data{ARTIFACT_SUFFIX}.csv"
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["f0", "f1", "f2", "f3", "f4", "y"])
        for row, lab in zip(X_test, y_test_raw):
            w.writerow([float(row[0]), float(row[1]), float(row[2]), float(row[3]), float(row[4]), int(lab)])
    print(f"Wrote {path}")


def save_weights_csv(model, ansatz_circuit, path: Path):
    vals = model.quantum_layer.weight.detach().cpu().numpy().reshape(-1)
    params = list(ansatz_circuit.parameters)
    assert len(vals) == len(params), (len(vals), len(params))
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["param_index", "param_name", "value"])
        for i, (p, v) in enumerate(zip(params, vals)):
            w.writerow([i, str(p), float(v)])
    print(f"Wrote {path}")


save_test_csv()


Wrote /Users/jkw/Documents/uni/axion/QC1/tests/ansatz_odra_evaluation/setup/test_data_trimmed_reverse_q0.csv


In [26]:
def train_and_save(name: str, ansatz_circuit: QuantumCircuit, loader):
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

    model = HybridModel(ansatz_circuit, N_QUBITS, MEASURE_QUBIT)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_acc = 0.0
    best_state = None

    print(f"\n=== Training {name} ===")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, loader, optimizer, loss_function)
        test_metrics = evaluate(model, X_test_tensor, y_test_tensor, loss_function)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"Epoch {epoch + 1:3d}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
                f"Test Acc: {test_metrics['accuracy']:.4f}"
            )
        if test_metrics["accuracy"] > best_acc:
            best_acc = test_metrics["accuracy"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    stem = name.lower().replace(" ", "_")
    torch.save(model.state_dict(), SETUP_DIR / f"weights_{stem}_final{ARTIFACT_SUFFIX}.pth")

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"Best test accuracy ({name}): {best_acc:.4f}")

    save_weights_csv(model, ansatz_circuit, SETUP_DIR / f"weights_{stem}{ARTIFACT_SUFFIX}.csv")
    torch.save(model.state_dict(), SETUP_DIR / f"weights_{stem}_best{ARTIFACT_SUFFIX}.pth")
    return model


odra_q = odra_ansatz(N_QUBITS, ANSATZ_DEPTH)
sim_q = simulator_ansatz(N_QUBITS, ANSATZ_DEPTH)
_exp = ansatz_trimmed_reverse_q0_param_count(N_QUBITS, ANSATZ_DEPTH)
assert len(list(odra_q.parameters)) == _exp
assert len(list(sim_q.parameters)) == _exp
print(
    f"Sanity: {len(list(odra_q.parameters))} parameters per ansatz "
    f"(expect {_exp} for n={N_QUBITS}, depth={ANSATZ_DEPTH})"
)

train_and_save("odra", odra_q, make_train_loader())
train_and_save("simulator", sim_q, make_train_loader())

print("\nDone. Artifacts in:", SETUP_DIR)


Sanity: 37 parameters per ansatz (expect 37 for n=5, depth=4)

=== Training odra ===
Trainable parameters: 37
Epoch   1/30 | Train Loss: 0.7236 | Test Acc: 0.8400
Epoch   5/30 | Train Loss: 0.5781 | Test Acc: 0.8618
Epoch  10/30 | Train Loss: 0.5769 | Test Acc: 0.8545
Epoch  15/30 | Train Loss: 0.5728 | Test Acc: 0.8291
Epoch  20/30 | Train Loss: 0.5718 | Test Acc: 0.8400
Epoch  25/30 | Train Loss: 0.5655 | Test Acc: 0.8182
Epoch  30/30 | Train Loss: 0.5675 | Test Acc: 0.8255
Best test accuracy (odra): 0.8764
Wrote /Users/jkw/Documents/uni/axion/QC1/tests/ansatz_odra_evaluation/setup/weights_odra_trimmed_reverse_q0.csv

=== Training simulator ===
Trainable parameters: 37
Epoch   1/30 | Train Loss: 0.7335 | Test Acc: 0.8327
Epoch   5/30 | Train Loss: 0.5704 | Test Acc: 0.8509
Epoch  10/30 | Train Loss: 0.5714 | Test Acc: 0.8436
Epoch  15/30 | Train Loss: 0.5707 | Test Acc: 0.8327
Epoch  20/30 | Train Loss: 0.5718 | Test Acc: 0.8436
Epoch  25/30 | Train Loss: 0.5667 | Test Acc: 0.8400
Ep